<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_5/All_Examples_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тема 4. Предобработка данных для RAG (Ingestion)

In [ ]:
# Установка только необходимых библиотек
!pip install pypdf pdfplumber

import os
import json
import logging
from datetime import datetime
from typing import Dict, List
from pathlib import Path

import pdfplumber
from pypdf import PdfReader

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def extract_pdf_text(file_path: str) -> str:
    """Извлекает текст из PDF с fallback."""
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            return text
    except Exception as e:
        logger.warning(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        logger.error(f"Не удалось извлечь текст из {file_path}: {e}")
        return ""

    return text.strip()


def clean_text(text: str) -> str:
    """Очищает текст от шума."""
    import re
    if not text:
        return ""
    # Удаляем управляющие символы
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_pdf_metadata(file_path: str) -> Dict:
    """Извлекает метаданные из PDF и файловой системы."""
    stat = os.stat(file_path)
    metadata = {
        "source": os.path.basename(file_path),
        "file_path": str(file_path),
        "doc_id": Path(file_path).stem,
        "created_date": datetime.fromtimestamp(stat.st_ctime).isoformat(),
        "modified_date": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "page_number": 0,
        "section": "",
        "tags": [],
        "department": "",
        "is_active": True
    }
    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            info = reader.metadata
            if info:
                metadata["author"] = str(info.get('/Author', ''))
                metadata["title"] = str(info.get('/Title', ''))
    except Exception as e:
        logger.warning(f"Не удалось извлечь метаданные PDF для {file_path}: {e}")
    return metadata


def process_pdf_directory(input_dir: str, output_json: str) -> List[Dict]:
    """Обрабатывает все PDF в директории и сохраняет результат в JSON."""
    results = []
    pdf_files = list(Path(input_dir).glob("**/*.pdf"))

    if not pdf_files:
        logger.warning(f"PDF файлы не найдены в {input_dir}")
        return results

    logger.info(f"Найдено {len(pdf_files)} PDF файлов")

    for pdf_path in pdf_files:
        logger.info(f"Обработка: {pdf_path}")
        text = extract_pdf_text(str(pdf_path))
        if not text:
            logger.warning(f"Пропущен (пустой текст): {pdf_path}")
            continue

        cleaned_text = clean_text(text)
        metadata = get_pdf_metadata(str(pdf_path))

        results.append({
            "text": cleaned_text,
            "metadata": metadata
        })
        logger.info(f"Добавлен: {pdf_path} (длина текста: {len(cleaned_text)} символов)")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    logger.info(f"Сохранено {len(results)} документов в {output_json}")
    return results


if __name__ == "__main__":
    process_pdf_directory(
        input_dir="./documents",
        output_json="./extracted_data.json"
    )

#Тема 5. Чанкинг (разбиение текста)

In [ ]:
# ================================================================
# Тема 5. Чанкинг (разбиение текста)
# Использует данные из extracted_data.json (результат раздела 4)
# ================================================================

import json
import re
from typing import List, Dict
from pathlib import Path

class RecursiveTextSplitter:
    """
    Рекурсивный сплиттер с иерархией разделителей.
    Разбивает текст на чанки, сохраняя структуру документа.
    """
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        # Иерархия разделителей: от крупных к мелким
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]

    def split_document(self, text: str, metadata: Dict) -> List[Dict]:
        """
        Разбивает текст на чанки и добавляет метаданные к каждому чанку.
        Возвращает список словарей: {"text": str, "metadata": dict}
        """
        chunks_text = self._split_text(text)
        chunks_with_meta = []
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            chunks_with_meta.append({
                "text": chunk_text,
                "metadata": chunk_meta
            })
        return chunks_with_meta

    def _split_text(self, text: str) -> List[str]:
        """Разбивает текст на чанки (внутренний метод)."""
        if not text:
            return []
        chunks = []
        current_chunk = []
        current_len = 0

        # Разбиваем текст рекурсивно по разделителям
        segments = self._split_by_separators(text, self.separators)

        for segment in segments:
            seg_len = len(segment)
            # Если текущий чанк + новый сегмент превышает max_size и чанк не пуст
            if current_len + seg_len > self.chunk_size and current_chunk:
                chunks.append("".join(current_chunk).strip())
                # Извлекаем overlap из предыдущего чанка
                overlap_text = self._get_overlap("".join(current_chunk), self.chunk_overlap)
                current_chunk = [overlap_text]
                current_len = len(overlap_text)

            current_chunk.append(segment)
            current_len += seg_len

        if current_chunk:
            chunks.append("".join(current_chunk).strip())

        return chunks

    def _split_by_separators(self, text: str, separators: List[str]) -> List[str]:
        """Рекурсивно разбивает текст по разделителям."""
        if not text:
            return []

        separator = separators[0]
        remaining_seps = separators[1:]

        if not remaining_seps:
            # Последний разделитель — пробел
            return text.split(separator)

        parts = text.split(separator)
        result = []
        for i, part in enumerate(parts):
            if len(part) <= self.chunk_size:
                result.append(part)
            else:
                # Рекурсивно разбиваем более мелким разделителем
                sub_parts = self._split_by_separators(part, remaining_seps)
                result.extend(sub_parts)

            if i < len(parts) - 1:
                result.append(separator)  # возвращаем разделитель обратно

        return result

    def _get_overlap(self, text: str, overlap_len: int) -> str:
        """Возвращает последние `overlap_len` символов текста."""
        return text[-overlap_len:] if len(text) > overlap_len else text


def load_extracted_data(json_path: str = "./extracted_data.json") -> List[Dict]:
    """Загружает данные, полученные в разделе 4."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def process_chunking(input_json: str = "./extracted_data.json",
                     output_json: str = "./chunks_data.json",
                     chunk_size: int = 500,
                     chunk_overlap: int = 50) -> List[Dict]:
    """
    Загружает данные из input_json, применяет чанкинг и сохраняет в output_json.
    Возвращает список всех чанков с метаданными.
    """
    # Загрузка данных из раздела 4
    documents = load_extracted_data(input_json)
    print(f"Загружено {len(documents)} документов.")

    splitter = RecursiveTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []

    for doc in documents:
        text = doc["text"]
        metadata = doc["metadata"]
        if not text:
            continue
        chunks = splitter.split_document(text, metadata)
        all_chunks.extend(chunks)

    # Сохранение результатов
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Создано {len(all_chunks)} чанков из {len(documents)} документов.")
    print(f"Результат сохранён в {output_json}")
    return all_chunks


def compare_chunk_sizes(input_json: str = "./extracted_data.json"):
    """
    Сравнивает три размера чанков на первом документе из загруженных данных.
    Выводит статистику для каждого размера.
    """
    documents = load_extracted_data(input_json)
    if not documents:
        print("Нет данных для эксперимента.")
        return

    sample_doc = documents[0]
    text = sample_doc["text"]
    metadata = sample_doc["metadata"]

    sizes = [200, 500, 1000]
    overlaps = [20, 50, 100]  # 10% от размера

    print("=" * 60)
    print(f"Эксперимент на документе: {metadata.get('source', 'unknown')}")
    print(f"Длина текста: {len(text)} символов")
    print("=" * 60)

    for size, overlap in zip(sizes, overlaps):
        splitter = RecursiveTextSplitter(chunk_size=size, chunk_overlap=overlap)
        chunks = splitter.split_document(text, metadata)
        print(f"\nРазмер чанка: {size}, overlap: {overlap}")
        print(f"  Количество чанков: {len(chunks)}")
        if chunks:
            print(f"  Пример первого чанка (первые 100 символов):")
            print(f"    {chunks[0]['text'][:100]}...")
        lengths = [len(c['text']) for c in chunks]
        print(f"  Средняя длина: {sum(lengths)/len(lengths):.0f} символов")
        print(f"  Минимальная: {min(lengths)}, максимальная: {max(lengths)}")


if __name__ == "__main__":
    # Шаг 1: применить чанкинг с параметрами по умолчанию
    chunks = process_chunking(
        input_json="./extracted_data.json",
        output_json="./chunks_data.json",
        chunk_size=500,
        chunk_overlap=50
    )

    # Шаг 2: провести сравнение размеров
    compare_chunk_sizes()

5.4.2. Использование LangChain (альтернативный вариант)

In [ ]:
!pip install langchain-text-splitters

In [ ]:
# ================================================================
# Тема 5. Чанкинг (разбиение текста) с использованием LangChain
# Альтернативный вариант к разделу 5.4.1
# Использует данные из extracted_data.json (результат раздела 4)
# ================================================================

import json
from typing import List, Dict

from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_extracted_data(json_path: str = "./extracted_data.json") -> List[Dict]:
    """Загружает данные, полученные в разделе 4."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def split_with_langchain(input_json: str = "./extracted_data.json",
                         output_json: str = "./chunks_data_langchain.json",
                         chunk_size: int = 500,
                         chunk_overlap: int = 50) -> List[Dict]:
    """
    Загружает данные из input_json, применяет чанкинг с помощью LangChain
    и сохраняет результат в output_json.
    Возвращает список всех чанков с метаданными.
    """
    # 1. Загрузка данных из раздела 4
    documents = load_extracted_data(input_json)
    print(f"Загружено {len(documents)} документов.")

    # 2. Настройка сплиттера LangChain
    # Используем RecursiveCharacterTextSplitter с теми же параметрами,
    # что и в нашей ручной реализации
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # Иерархия разделителей: от крупных к мелким
        separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
        length_function=len,  # считаем длину в символах
    )

    all_chunks = []

    # 3. Обработка каждого документа
    for doc in documents:
        text = doc["text"]
        metadata = doc["metadata"]

        if not text:
            continue

        # Разбиваем текст на чанки с помощью LangChain
        chunks_text = splitter.split_text(text)

        # Добавляем метаданные к каждому чанку
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            all_chunks.append({
                "text": chunk_text,
                "metadata": chunk_meta
            })

    # 4. Сохранение результатов
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Создано {len(all_chunks)} чанков из {len(documents)} документов.")
    print(f"Результат сохранён в {output_json}")
    return all_chunks


def compare_chunk_sizes_langchain(input_json: str = "./extracted_data.json"):
    """
    Сравнивает три размера чанков на первом документе из загруженных данных.
    Использует LangChain для сплиттинга.
    """
    documents = load_extracted_data(input_json)
    if not documents:
        print("Нет данных для эксперимента.")
        return

    sample_doc = documents[0]
    text = sample_doc["text"]
    metadata = sample_doc["metadata"]

    sizes = [200, 500, 1000]
    overlaps = [20, 50, 100]  # 10% от размера

    print("=" * 60)
    print(f"Эксперимент (LangChain) на документе: {metadata.get('source', 'unknown')}")
    print(f"Длина текста: {len(text)} символов")
    print("=" * 60)

    for size, overlap in zip(sizes, overlaps):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=overlap,
            separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
            length_function=len,
        )
        chunks_text = splitter.split_text(text)
        print(f"\nРазмер чанка: {size}, overlap: {overlap}")
        print(f"  Количество чанков: {len(chunks_text)}")
        if chunks_text:
            print(f"  Пример первого чанка (первые 100 символов):")
            print(f"    {chunks_text[0][:100]}...")
        lengths = [len(c) for c in chunks_text]
        if lengths:
            print(f"  Средняя длина: {sum(lengths)/len(lengths):.0f} символов")
            print(f"  Минимальная: {min(lengths)}, максимальная: {max(lengths)}")


if __name__ == "__main__":
    # Шаг 1: применить чанкинг с параметрами по умолчанию
    chunks = split_with_langchain(
        input_json="./extracted_data.json",
        output_json="./chunks_data_langchain.json",
        chunk_size=500,
        chunk_overlap=50
    )

    # Шаг 2: провести сравнение размеров
    compare_chunk_sizes_langchain()

# Тема 6. Векторизация и эмбеддинги

6.2.2. BGE (BAAI/bge)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Загрузка модели
model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# Важно: для BGE нужно использовать префиксы!
# Для запросов (queries) — "query: "
# Для документов (passages) — "passage: "

queries = [
    "query: Какие налоги платят самозанятые?",
    "query: Ставка налога на прибыль"
]

documents = [
    "passage: Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "passage: Ставка налога на прибыль составляет 20%.",
    "passage: Самозанятые платят налог на профессиональный доход."
]

# Генерация эмбеддингов
query_embeddings = model.encode(queries, normalize_embeddings=True)
doc_embeddings = model.encode(documents, normalize_embeddings=True)

# Вычисление косинусного сходства
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(query_embeddings, doc_embeddings)

print("Матрица сходства (запросы × документы):")
print(scores)

6.2.3. OpenAI Embeddings

In [ ]:
!pip install openai
import openai
import numpy as np
from typing import List

# Установите ваш API-ключ
openai.api_key = "your-api-key-here"

def get_openai_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> np.ndarray:
    """
    Генерирует эмбеддинги через OpenAI API.
    """
    # Для моделей text-embedding-3 нужно уменьшить размерность (опционально)
    # dimensions=1024  # можно уменьшить до 1024 для экономии
    response = openai.embeddings.create(
        model=model,
        input=texts,
        # dimensions=1024  # раскомментируйте, если хотите уменьшить размерность
    )
    embeddings = np.array([item.embedding for item in response.data])
    return embeddings

# Пример использования
texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%."
]

embeddings = get_openai_embeddings(texts, model="text-embedding-3-small")
print(f"Размерность: {embeddings.shape[1]}")
print(f"Форма: {embeddings.shape}")

6.2.4. Русскоязычные модели

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Работающая русская модель
model = SentenceTransformer('cointegrated/rubert-tiny2')

texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%.",
    "Самозанятые платят налог на профессиональный доход."
]

embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"Размерность: {embeddings.shape[1]}")
print(f"Вектор для первого текста (первые 5 значений): {embeddings[0][:5]}")

6.3.2. Генерация эмбеддингов с помощью sentence‑transformers

In [ ]:
!pip install --upgrade sentence-transformers pillow transformers
# Установка только нужных библиотек для извлечения текста
!pip install pypdf pdfplumber

# Установка sentence-transformers и фикс Pillow
!pip install sentence-transformers
!pip uninstall pillow -y
!pip install pillow==10.4.0

In [ ]:
# Теперь импорты должны работать
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class EmbeddingGenerator:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str = "cpu"):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

# Проверка
generator = EmbeddingGenerator()
texts = ["Пример текста", "Ещё один документ"]
embeddings = generator.encode_batch(texts)
print(embeddings.shape)

6.3.3. Кэширование эмбеддингов на диск

In [ ]:
import pickle
import os
import numpy as np
from typing import Dict, List, Any
from sentence_transformers import SentenceTransformer

class EmbeddingGenerator:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str = None):
        # Если device не указан, определяем автоматически
        if device is None:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

class EmbeddingCache:
    def __init__(self, cache_dir: str = "./embedding_cache"):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def get_cache_path(self, model_name: str, chunk_id: int) -> str:
        return os.path.join(self.cache_dir, f"{model_name}_{chunk_id}.pkl")

    def save_embeddings(self, model_name: str, chunk_id: int, embeddings: np.ndarray, metadata: Dict = None):
        cache_path = self.get_cache_path(model_name, chunk_id)
        data = {"embeddings": embeddings, "metadata": metadata}
        with open(cache_path, "wb") as f:
            pickle.dump(data, f)

    def load_embeddings(self, model_name: str, chunk_id: int) -> Dict:
        cache_path = self.get_cache_path(model_name, chunk_id)
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                return pickle.load(f)
        return None

    def exists(self, model_name: str, chunk_id: int) -> bool:
        return os.path.exists(self.get_cache_path(model_name, chunk_id))

# Пример использования
generator = EmbeddingGenerator("all-MiniLM-L6-v2")  # device определится автоматически
cache = EmbeddingCache("./embedding_cache")

texts = ["Это текст 1", "Это текст 2"]
chunk_id = 1

if cache.exists(generator.model_name, chunk_id):
    data = cache.load_embeddings(generator.model_name, chunk_id)
    embeddings = data["embeddings"]
    print("Эмбеддинги загружены из кэша")
else:
    embeddings = generator.encode_batch(texts)
    cache.save_embeddings(generator.model_name, chunk_id, embeddings, {"texts": texts})
    print("Эмбеддинги сгенерированы и сохранены в кэш")

print(f"Форма эмбеддингов: {embeddings.shape}")

Тема 7. Векторные базы данных

In [ ]:
!pip install chromadb

In [ ]:
import chromadb
from chromadb.config import Settings
import json
from sentence_transformers import SentenceTransformer

def load_documents(json_path: str = "./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_metadata(metadata: dict) -> dict:
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, list):
            if len(value) == 0:
                continue
            value = [v for v in value if v is not None]
            if len(value) == 0:
                continue
        elif value is None or value == "":
            continue
        cleaned[key] = value
    return cleaned

# 1. Инициализация
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(anonymized_telemetry=False)
)

collection = client.get_or_create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}
)

# 2. Загрузка данных
documents = load_documents("./extracted_data.json")

texts = [doc["text"] for doc in documents]
metadatas = [clean_metadata(doc["metadata"]) for doc in documents]
ids = [f"doc_{i:04d}" for i in range(len(documents))]

# 3. Генерация эмбеддингов (если их нет)
model = SentenceTransformer("cointegrated/rubert-tiny2")
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()

# 4. Добавление
collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)
print(f"✅ Добавлено {len(documents)} документов.")

# 5. Поиск БЕЗ ФИЛЬТРА
query = "налог на прибыль"
query_emb = model.encode([query], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=query_emb,
    n_results=3
)

print(f"\n🔍 Найдено: {len(results['documents'][0])} документов")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  - {doc[:150]}... (источник: {meta.get('source', 'unknown')})")


### 7.6.2. FAISS – максимальная скорость, метаданные отдельно




In [ ]:
# ================================================================
# FAISS: исправленная версия (адаптивная)
# ================================================================

import faiss
import numpy as np
import json
from sentence_transformers import SentenceTransformer

def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Загрузка данных
docs = load_documents()
texts = [d["text"] for d in docs]
metadatas = [d["metadata"] for d in docs]

if not texts:
    print("Нет документов для индексации")
    exit()

model = SentenceTransformer("cointegrated/rubert-tiny2")
embeddings = model.encode(texts, normalize_embeddings=True).astype('float32')
dim = embeddings.shape[1]
n_vectors = len(embeddings)

# Выбор типа индекса в зависимости от количества векторов
if n_vectors < 100:
    # Для малого числа документов используем точный поиск (Flat)
    index = faiss.IndexFlatIP(dim)  # IP = Inner Product (для нормализованных векторов даёт косинусное сходство)
    index.add(embeddings)
    print(f"Используется IndexFlatIP (точный поиск) для {n_vectors} векторов")
else:
    # Для больших данных – IVF
    nlist = min(100, max(1, n_vectors // 10))
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist)
    index.train(embeddings)
    index.add(embeddings)
    index.nprobe = min(5, nlist)
    print(f"Используется IndexIVFFlat с nlist={nlist}, nprobe={index.nprobe}")

# Поиск
query = "налог на прибыль"
query_emb = model.encode([query], normalize_embeddings=True).astype('float32')

distances, indices = index.search(query_emb, min(3, n_vectors))

print("Результаты поиска FAISS:")
for idx, dist in zip(indices[0], distances[0]):
    if idx >= 0 and idx < len(texts):
        print(f"  - {metadatas[idx].get('source', 'unknown')}: {texts[idx][:150]}... (сходство: {dist:.4f})")
    else:
        print(f"  - индекс {idx} вне диапазона")



### 7.6.3. Pinecone – облачный сервис (требуется API-ключ)


In [ ]:
!pip install --upgrade pinecone

In [ ]:
# ================================================================
# PINECONE – ФИНАЛЬНЫЙ РАБОЧИЙ КОД (регион us-east-1)
# ================================================================

import json
import time
from sentence_transformers import SentenceTransformer
import pinecone
from google.colab import userdata

# 1. Загрузка документов
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# 2. Инициализация Pinecone
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)

# 3. Параметры индекса (исправленный регион)
index_name = "rag-docs"
dimension = 312
metric = "cosine"
cloud = "aws"          # Используем AWS
region = "us-east-1"   # Стандартный регион для новых аккаунтов

# 4. Создание индекса (если не существует)
existing_indexes = [idx.name for idx in pc.list_indexes()]
if index_name not in existing_indexes:
    print(f"Создание индекса {index_name} в регионе {region}...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=pinecone.ServerlessSpec(cloud=cloud, region=region)
    )
    # Ожидание готовности (до 2 минут)
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(10)
        print("Ожидание готовности индекса...")
    print("✅ Индекс готов.")
else:
    print(f"ℹ️ Индекс {index_name} уже существует.")

index = pc.Index(index_name)

# 5. Загрузка данных и вставка
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

batch_size = 100
total = len(docs)
for i in range(0, total, batch_size):
    batch = docs[i:i+batch_size]
    texts = [d["text"] for d in batch]
    emb = model.encode(texts, normalize_embeddings=True).tolist()
    metas = [d["metadata"] for d in batch]
    ids = [f"doc_{i+j}" for j in range(len(batch))]
    index.upsert(vectors=list(zip(ids, emb, metas)))
    print(f"Добавлено {len(batch)} из {total}")

print(f"✅ Всего добавлено {total} документов.")

# 6. Поиск
query = "налог на прибыль"
qe = model.encode([query], normalize_embeddings=True).tolist()
res = index.query(vector=qe, top_k=3, include_metadata=True)

print("\n🔍 Результаты поиска:")
if res['matches']:
    for match in res['matches']:
        src = match['metadata'].get('source', 'unknown')
        print(f"  - {src}: сходство {match['score']:.4f}")
else:
    print("  Ничего не найдено.")


### 7.6.4. Weaviate – гибридный поиск (BM25 + вектор)




In [ ]:
# ================================================================
# Weaviate – подключение через Colab Secrets (Исправлено для v4)
# ================================================================

!pip install weaviate-client sentence-transformers -q

import json
from sentence_transformers import SentenceTransformer
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery
from weaviate.classes.init import Auth
from google.colab import userdata

# ---- Загрузка секретов ----
WEAVIATE_URL = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')

if not WEAVIATE_URL or not WEAVIATE_API_KEY:
    raise ValueError(
        "❌ Секреты не найдены!\n"
        "Добавьте их в панели 🔑 Secrets:\n"
        "  - WEAVIATE_URL = https://atpl3xhmr8aaimzordfnua.c0.eu-central-1.aws.weaviate.cloud\n"
        "  - WEAVIATE_API_KEY = ваш_секретный_ключ"
    )

# ---- Загрузка документов ----
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- Подключение ----
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY), # Современный способ аутентификации
)

# ---- Создание коллекции ----
if client.collections.exists("Document"):
    client.collections.delete("Document")

collection = client.collections.create(
    name="Document",
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="author", data_type=DataType.TEXT),
    ],
    vectorizer_config=Configure.Vectorizer.none(), # Явное указание отсутствия встроенного векизатора
)

# ---- Загрузка и вставка ----
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

with collection.batch.fixed_size(batch_size=100) as batch:
    for doc in docs:
        text = doc["text"]
        emb = model.encode(text, normalize_embeddings=True).tolist()
        properties = {
            "text": text,
            "source": doc["metadata"].get("source", ""),
            "author": doc["metadata"].get("author", ""),
        }
        batch.add_object(properties=properties, vector=emb)

print(f"✅ Добавлено {len(docs)} документов.")

# ---- Векторный поиск ----
query = "налог на прибыль"
# Передаем строку, чтобы получить 1D массив (один вектор), а не список из одного вектора
qe = model.encode(query, normalize_embeddings=True).tolist()

vector_results = collection.query.near_vector(
    near_vector=qe,         # Передается сам вектор
    distance=0.5,           # Отдельный параметр для фильтра по дистанции
    limit=3,
    return_properties=["text", "source"],
    return_metadata=MetadataQuery(distance=True) # Запрашиваем distance для obj.metadata
)

print("\n🔍 Weaviate (векторный поиск):")
for obj in vector_results.objects:
    print(f"  - {obj.properties['source']}: {obj.properties['text'][:150]}... (distance: {obj.metadata.distance:.4f})")

# ---- Гибридный поиск ----
hybrid_results = collection.query.hybrid(
    query=query,
    alpha=0.5,
    limit=3,
    return_properties=["text", "source"],
    return_metadata=MetadataQuery(score=True) # Запрашиваем score для obj.metadata
)

print("\n🔍 Weaviate (гибридный поиск):")
for obj in hybrid_results.objects:
    print(f"  - {obj.properties['source']}: {obj.properties['text'][:150]}... (score: {obj.metadata.score:.4f})")

client.close()


### 7.6.5. Qdrant – гибкая фильтрация payload




In [ ]:
!pip install qdrant-client sentence-transformers

# ================================================================
# Qdrant – локальный режим (без сервера)
# ================================================================

import json
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# ---- Загрузка данных ----
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- Подключение к локальной БД (хранилище на диске) ----
client = QdrantClient(path="./qdrant_data")  # данные сохранятся в папку qdrant_data

collection_name = "documents"

# ---- Пересоздание коллекции ----
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=312, distance=Distance.COSINE)
)

# ---- Загрузка документов ----
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

# ---- Подготовка точек ----
points = []
for i, doc in enumerate(docs):
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    points.append(PointStruct(
        id=i,
        vector=emb,
        payload={
            "text": doc["text"],
            "source": doc["metadata"].get("source", ""),
            "author": doc["metadata"].get("author", ""),
        }
    ))

# ---- Вставка данных ----
client.upsert(collection_name=collection_name, points=points)
print(f"✅ Добавлено {len(points)} документов.")

# ---- Поиск ----
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

# Без фильтра (просто поиск)
results = client.search(
    collection_name=collection_name,
    query_vector=qe,
    limit=3,
    with_payload=True,
)

print("\n🔍 Qdrant (поиск без фильтра):")
for hit in results:
    print(f"  - {hit.payload['source']}: {hit.payload['text'][:150]}... (score: {hit.score:.4f})")

# ---- Поиск с фильтром ----
from qdrant_client.models import Filter, FieldCondition, MatchValue

filtered_results = client.search(
    collection_name=collection_name,
    query_vector=qe,
    limit=3,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchValue(value="report_2024.pdf")  # замените на существующий файл
            )
        ]
    ),
    with_payload=True,
)

print("\n🔍 Qdrant (с фильтром по источнику):")
for hit in filtered_results:
    print(f"  - {hit.payload['source']}: {hit.payload['text'][:150]}... (score: {hit.score:.4f})")


### 7.6.6. Milvus – масштабирование до миллиардов




In [ ]:
# ================================================================
# Milvus: распределённый поиск
# ================================================================
!pip install pymilvus sentence-transformers

from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType, utility

connections.connect("default", host="localhost", port="19530")
collection_name = "documents"

# Безопасное удаление старой коллекции
if utility.has_collection(collection_name):
    Collection(collection_name).drop()

fields = [
    FieldSchema("id", DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema("text", DataType.VARCHAR, max_length=65535),
    FieldSchema("source", DataType.VARCHAR, max_length=255),
    # Если нужно поле author, раскомментируйте строку ниже:
    # FieldSchema("author", DataType.VARCHAR, max_length=255),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=312),
]
schema = CollectionSchema(fields)
collection = Collection(collection_name, schema)

docs = load_documents()
model = SentenceTransformer("cointegrated/rubert-tiny2")

data = []
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    data.append({
        "text": doc["text"],
        "source": doc["metadata"].get("source", ""),
        # "author": doc["metadata"].get("author", ""), # Если добавите поле в схему
        "embedding": emb,
    })

# Вставка данных
collection.insert(data)

# Создание индекса (обязательно для быстрого поиска)
collection.create_index(
    "embedding",
    {"metric_type": "COSINE", "index_type": "IVF_FLAT", "params": {"nlist": 128}}
)

# Загрузка коллекции в память (обязательно перед поиском)
collection.load()

query = "налог на прибыль"
# ИСПРАВЛЕНИЕ: Убираем скобки вокруг query, чтобы получить 1D-массив (один вектор)
qe = model.encode(query, normalize_embeddings=True).tolist()

# ИСПРАВЛЕНИЕ: Передаем [qe], чтобы получился корректный 2D-массив для поиска
results = collection.search(
    [qe],
    "embedding",
    {"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    output_fields=["text", "source"] # Сюда же можно добавить "author", если он есть в схеме
)

print("\n🔍 Milvus (векторный поиск):")
# results содержит список объектов Hits (по одному на каждый запрос)
for hits in results:
    for hit in hits:
        # Поля из output_fields доступны через hit.entity
        text = hit.entity.get("text")
        source = hit.entity.get("source")
        print(f"  - {source}: {text[:150]}... (distance: {hit.distance:.4f})")


### 7.6.7. LanceDB – хранение на диске (экономия RAM)




In [ ]:
# ================================================================
# LanceDB: данные на диске, индекс не в памяти
# ================================================================
!pip install lancedb pandas sentence-transformers

# !pip install lancedb pandas sentence-transformers -q

import lancedb
import pandas as pd
from sentence_transformers import SentenceTransformer

db = lancedb.connect("./lancedb_data")
docs = load_documents()
model = SentenceTransformer("cointegrated/rubert-tiny2")

data = []
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    data.append({
        "text": doc["text"],
        "source": doc["metadata"].get("source", ""),
        "vector": emb, # LanceDB по умолчанию ищет по полю "vector"
    })

df = pd.DataFrame(data)
table = db.create_table("documents", data=df, mode="overwrite")

# СОЗДАНИЕ ИНДЕКСА (ОПЦИОНАЛЬНО)
# Если у вас БОЛЬШЕ 256 документов, можно создать IVF_PQ индекс для ускорения на больших объемах
if len(docs) >= 256:
    table.create_index(
        metric="cosine",
        index_type="IVF_PQ",
        num_partitions=10,
        num_sub_vectors=16
    )
# Если документов мало, LanceDB сам сделает быстрый flat-поиск без индекса!

query = "налог на прибыль"
# ИСПРАВЛЕНИЕ: Убираем скобки, чтобы получить 1D-массив (один вектор)
qe = model.encode(query, normalize_embeddings=True).tolist()

# ПОИСК
# Явно указываем метрику "cosine", чтобы получить косинусное расстояние
results = table.search(qe).metric("cosine").limit(3).to_pandas()

print("\n🔍 LanceDB (поиск с диска):")
# Метод to_pandas() автоматически добавляет колонку '_distance'
for _, row in results.iterrows():
    print(f"  - {row['source']}: {row['text'][:150]}... (distance: {row['_distance']:.4f})")


### 7.6.8. PgVector – родной SQL и ACID в PostgreSQL




In [ ]:
# ================================================================
# PgVector: векторный поиск внутри PostgreSQL
# ================================================================
!pip install psycopg2-binary sentence-transformers pgvector


# !pip install psycopg2-binary sentence-transformers pgvector -q

import psycopg2
from pgvector.psycopg2 import register_vector # ВАЖНО: импортируем адаптер
from sentence_transformers import SentenceTransformer

# 1. Подключение
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="password",
    host="localhost"
)
# 2. Регистрация типа vector для psycopg2
# Теперь psycopg2 умеет автоматически переводить Python-списки в тип vector
register_vector(conn)
cur = conn.cursor()

# 3. Создание таблицы и расширения
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("DROP TABLE IF EXISTS documents;")
cur.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        text TEXT,
        source TEXT,
        embedding vector(312)
    );
""")

docs = load_documents()
model = SentenceTransformer("cointegrated/rubert-tiny2")

# 4. Вставка данных
print(f"📥 Вставка {len(docs)} документов...")
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    # Благодаря register_vector, мы можем передавать emb (list) напрямую!
    cur.execute(
        "INSERT INTO documents (text, source, embedding) VALUES (%s, %s, %s)",
        (doc["text"], doc["metadata"].get("source", ""), emb)
    )
conn.commit()

# 5. Создание индекса HNSW для ускорения поиска
# Рекомендуется указывать параметры m и ef_construction для лучших результатов
cur.execute("""
    CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
conn.commit()

query = "налог на прибыль"
# ИСПРАВЛЕНИЕ: Убираем скобки, чтобы получить 1D-массив (один вектор)
qe = model.encode(query, normalize_embeddings=True).tolist()

# 6. Поиск
# Оператор <=> вычисляет косинусное расстояние (cosine distance)
# 1 - (embedding <=> %s) превращает расстояние в сходство (similarity от 0 до 1)
cur.execute("""
    SELECT text, source, 1 - (embedding <=> %s) AS similarity
    FROM documents
    ORDER BY embedding <=> %s
    LIMIT 3;
""", (qe, qe))

results = cur.fetchall()
print("\n🔍 PgVector (SQL поиск):")
for text, source, similarity in results:
    print(f"  - {source}: {text[:150]}... (similarity: {similarity:.4f})")

# 7. ОБЯЗАТЕЛЬНО закрываем ресурсы
cur.close()
conn.close()

`
